# GPT 라벨러 신뢰도 검증

수동 라벨(Manual)과 GPT 자동 라벨(GPT)의 일치도를 두 가지 방식으로 평가합니다.

1. **Jaccard 유사도**: 라벨 세트 비교 (직관적)
2. **확장방식**: 51개 속성-감성 쌍 비교 (엄밀함)

## 라이브러리 임포트

In [ ]:
import json
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

print('라이브러리 로드 완료')

## 데이터 로드

In [ ]:
def load_jsonl(filepath):
    data = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            review_id = item['id']
            data[review_id] = item['annotation']
    return data

#수동테스트셋(데이터셋은 포함하지 말래서 어케할까요)
manual_data = load_jsonl(r'C:\Users\WONHO\Desktop\SSU\26-1\datascience\project\fianl_manual_label_test_set.jsonl')
#gpt 라벨링 결과_14741개랑 비교
gpt_data = load_jsonl(r'c:\Users\WONHO\Desktop\SSU\26-1\datascience\project\labelled_AI\steam_reviews_pre_labeled_GPT.jsonl')

matching_ids = sorted(set(manual_data.keys()) & set(gpt_data.keys()))

print(f'Manual 데이터: {len(manual_data):,}개')
print(f'GPT 데이터: {len(gpt_data):,}개')
print(f'일치하는 리뷰: {len(matching_ids)}개')

---
# 방식 1: Jaccard 유사도 기반 부분일치 분석

## Jaccard 유사도 계산

In [ ]:
def calculate_jaccard_similarity(set1, set2):
    """두 라벨 세트의 Jaccard 유사도 계산"""
    labels1 = set(tuple(label) for label in set1)
    labels2 = set(tuple(label) for label in set2)
    
    if len(labels1) == 0 and len(labels2) == 0:
        return 1.0
    
    intersection = len(labels1 & labels2)
    union = len(labels1 | labels2)
    
    if union == 0:
        return 1.0
    
    return intersection / union

def calculate_dissimilarity(set1, set2):
    """Jaccard distance = 1 - Jaccard similarity"""
    return 1 - calculate_jaccard_similarity(set1, set2)

jaccard_similarities = []
dissimilarities = []
detailed_results = []

for review_id in matching_ids:
    manual_labels = manual_data[review_id]
    gpt_labels = gpt_data[review_id]
    
    jaccard = calculate_jaccard_similarity(manual_labels, gpt_labels)
    dissimilarity = calculate_dissimilarity(manual_labels, gpt_labels)
    
    jaccard_similarities.append(jaccard)
    dissimilarities.append(dissimilarity)
    
    detailed_results.append({
        'id': review_id,
        'manual_labels': manual_labels,
        'gpt_labels': gpt_labels,
        'manual_count': len(manual_labels),
        'gpt_count': len(gpt_labels),
        'jaccard': jaccard,
        'dissimilarity': dissimilarity
    })

results_df = pd.DataFrame(detailed_results)
print('처음 10개 리뷰의 Jaccard 유사도:')
print(results_df[['id', 'manual_count', 'gpt_count', 'jaccard']].head(10))

## Jaccard 통계 분석

In [ ]:
mean_jaccard = np.mean(jaccard_similarities)
std_jaccard = np.std(jaccard_similarities)
median_jaccard = np.median(jaccard_similarities)

perfect_match = sum(1 for j in jaccard_similarities if j == 1.0)
partial_match = sum(1 for j in jaccard_similarities if 0 < j < 1.0)
no_match = sum(1 for j in jaccard_similarities if j == 0.0)

print('\n' + '='*70)
print('JACCARD 유사도 통계')
print('='*70)
print(f'평균 (Mean):        {mean_jaccard:.4f}')
print(f'중앙값 (Median):    {median_jaccard:.4f}')
print(f'표준편차 (Std):     {std_jaccard:.4f}')
print(f'최소값 (Min):       {min(jaccard_similarities):.4f}')
print(f'최대값 (Max):       {max(jaccard_similarities):.4f}')
print(f'\n' + '='*70)
print('일치도별 분포')
print('='*70)
print(f'완전 일치 (Jaccard=1.0):        {perfect_match}개 ({100*perfect_match/len(jaccard_similarities):.1f}%)')
print(f'부분 일치 (0 < Jaccard < 1.0): {partial_match}개 ({100*partial_match/len(jaccard_similarities):.1f}%)')
print(f'불일치 (Jaccard=0.0):         {no_match}개 ({100*no_match/len(jaccard_similarities):.1f}%)')

## Jaccard 기반 Krippendorff's Alpha

In [ ]:
D_o_jaccard = np.mean(dissimilarities)

all_labels_combined = []
for review_id in matching_ids:
    all_labels_combined.extend(manual_data[review_id])
    all_labels_combined.extend(gpt_data[review_id])

expected_dissimilarities = []
for i in range(len(all_labels_combined)):
    for j in range(i+1, len(all_labels_combined)):
        label1 = tuple(all_labels_combined[i])
        label2 = tuple(all_labels_combined[j])
        if label1 == label2:
            expected_dissimilarities.append(0.0)
        else:
            expected_dissimilarities.append(1.0)

D_e_jaccard = np.mean(expected_dissimilarities) if expected_dissimilarities else 0
alpha_jaccard = 1 - (D_o_jaccard / D_e_jaccard) if D_e_jaccard > 0 else 1.0

print('\n' + '='*70)
print("KRIPPENDORFF'S ALPHA (Jaccard 유사도 기반)")
print('='*70)
print(f'관찰된 불일치도 (D_o): {D_o_jaccard:.4f}')
print(f'기대 불일치도 (D_e):  {D_e_jaccard:.4f}')
print(f'\nKrippendorff Alpha: {alpha_jaccard:.4f}')

if alpha_jaccard >= 0.81:
    interpretation_jaccard = 'Almost Perfect Agreement (거의 완벽한 일치)'
elif alpha_jaccard >= 0.61:
    interpretation_jaccard = 'Substantial Agreement (상당한 일치)'
elif alpha_jaccard >= 0.41:
    interpretation_jaccard = 'Moderate Agreement (중간 정도의 일치)'
else:
    interpretation_jaccard = 'Fair/Weak Agreement (약한 일치)'

print(f'해석: {interpretation_jaccard}')

---
# 방식 2: 51개 속성-감성 쌍 확장 방식

## 51개 속성-감성 쌍 생성

In [ ]:
attributes = [
    '최적화#프레임',
    '최적화#조작감',
    '시스템#버그',
    '콘텐츠#볼륨',
    '콘텐츠#몰입도',
    '시스템#밸런스',
    '시스템#독창성',
    '시스템#자유도',
    'UX#그래픽',
    'UX#캐릭터디자인',
    'UX#사운드',
    '스토리#내러티브',
    '운영#핵/치트',
    '운영#업데이트',
    '콘텐츠#난이도',
    '시스템#진입장벽',
    '콘텐츠#피로도'
]
sentiments = ['positive', 'negative', 'neutral']

pairs = list(product(attributes, sentiments))
pair_idx = {(attr, sent): i for i, (attr, sent) in enumerate(pairs)}

print(f'총 쌍의 개수: {len(pairs)}')
print(f'속성: {len(attributes)}개, 감성: {len(sentiments)}개')

## 이진 변환 (51쌍 → 0/1)

In [ ]:
def to_binary(annotations):
    binary = [0] * len(pairs)
    for ann in annotations:
        if len(ann) >= 2 and (ann[0], ann[1]) in pair_idx:
            binary[pair_idx[(ann[0], ann[1])]] = 1
    return binary

manual_b = {rid: to_binary(manual_data[rid]) for rid in matching_ids}
gpt_b = {rid: to_binary(gpt_data[rid]) for rid in matching_ids}

print('이진 변환 완료')

## 확장방식 일치도 분석

In [ ]:
agreement_rates = []
for rid in matching_ids:
    matches = sum(1 for m, g in zip(manual_b[rid], gpt_b[rid]) if m == g)
    rate = 100 * matches / len(pairs)
    agreement_rates.append(rate)

print('\n' + '='*70)
print('51쌍 확장방식 일치도')
print('='*70)
print(f'평균: {np.mean(agreement_rates):.2f}%')
print(f'최소: {min(agreement_rates):.2f}%')
print(f'최대: {max(agreement_rates):.2f}%')

## 확장방식 Krippendorff's Alpha

In [ ]:
D_o_exp_sum = 0
for rid in matching_ids:
    for m, g in zip(manual_b[rid], gpt_b[rid]):
        if m != g:
            D_o_exp_sum += 1

total = len(matching_ids) * len(pairs)
D_o_exp = D_o_exp_sum / total

all_values = []
for rid in matching_ids:
    all_values.extend(manual_b[rid])
    all_values.extend(gpt_b[rid])

count_0 = all_values.count(0)
count_1 = all_values.count(1)
p_0 = count_0 / len(all_values)
p_1 = count_1 / len(all_values)
D_e_exp = 2 * p_0 * p_1

alpha_exp = 1 - (D_o_exp / D_e_exp) if D_e_exp > 0 else 1.0

print('\n' + '='*70)
print('KRIPPENDORFF ALPHA (51-쌍 확장방식)')
print('='*70)
print(f'D_o (관찰된 불일치): {D_o_exp:.4f}')
print(f'D_e (기대 불일치):  {D_e_exp:.4f}')
print(f'\nAlpha: {alpha_exp:.4f}')

if alpha_exp >= 0.81:
    interpretation_exp = 'Almost Perfect Agreement'
elif alpha_exp >= 0.61:
    interpretation_exp = 'Substantial Agreement'
elif alpha_exp >= 0.41:
    interpretation_exp = 'Moderate Agreement'
else:
    interpretation_exp = 'Fair/Weak Agreement'

print(f'해석: {interpretation_exp}')

## 혼동 행렬 및 성능 지표

In [ ]:
tp = fp = fn = tn = 0
for rid in matching_ids:
    for m, g in zip(manual_b[rid], gpt_b[rid]):
        if m == 1 and g == 1:
            tp += 1
        elif m == 0 and g == 1:
            fp += 1
        elif m == 1 and g == 0:
            fn += 1
        else:
            tn += 1

prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
acc = (tp + tn) / (tp + fp + fn + tn)

print('\n' + '='*70)
print('CONFUSION MATRIX')
print('='*70)
print(f'TP (둘 다 1):   {tp:,}')
print(f'FP (M=0, G=1): {fp:,}')
print(f'FN (M=1, G=0): {fn:,}')
print(f'TN (둘 다 0):   {tn:,}')
print(f'\n성능 지표:')
print(f'정밀도 (Precision): {prec:.4f}')
print(f'재현율 (Recall):    {rec:.4f}')
print(f'F1 점수:           {f1:.4f}')
print(f'정확도 (Accuracy):  {acc:.4f}')

## 속성별 일치도

In [ ]:
attr_agreement = {}
for attr in attributes:
    indices = [pair_idx[(attr, sent)] for sent in sentiments]
    matches = 0
    for rid in matching_ids:
        for idx in indices:
            if manual_b[rid][idx] == gpt_b[rid][idx]:
                matches += 1
    attr_agreement[attr] = 100 * matches / (len(matching_ids) * len(sentiments))

print('\n' + '='*70)
print('ATTRIBUTE-WISE AGREEMENT (상위 10개)')
print('='*70)
for attr, rate in sorted(attr_agreement.items(), key=lambda x: -x[1])[:10]:
    print(f'{attr:25s}: {rate:6.2f}%')

---
# 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Jaccard 분포
axes[0, 0].hist(jaccard_similarities, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(mean_jaccard, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_jaccard:.3f}')
axes[0, 0].axvline(median_jaccard, color='green', linestyle='--', linewidth=2, label=f'Median: {median_jaccard:.3f}')
axes[0, 0].set_xlabel('Jaccard Similarity')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Jaccard Similarity Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 일치도 분포 (파이차트)
agreement_counts = [perfect_match, partial_match, no_match]
agreement_labels = [f'Perfect\nMatch\n(n={perfect_match})', f'Partial\nMatch\n(n={partial_match})', f'No\nMatch\n(n={no_match})']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0, 1].pie(agreement_counts, labels=agreement_labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0, 1].set_title('Agreement Category Distribution')

# Box plot
axes[1, 0].boxplot([jaccard_similarities], labels=['Jaccard Similarity'])
axes[1, 0].set_ylabel('Value')
axes[1, 0].set_title('Box Plot of Jaccard Similarity')
axes[1, 0].grid(True, alpha=0.3)

# Manual vs GPT 라벨 개수
axes[1, 1].scatter(results_df['manual_count'], results_df['gpt_count'], alpha=0.6, s=60, color='purple')
axes[1, 1].set_xlabel('Manual Label Count')
axes[1, 1].set_ylabel('GPT Label Count')
axes[1, 1].set_title('Manual vs GPT Label Counts')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# 최종 비교 및 요약

In [ ]:
print('\n' + '='*80)
print('최종 신뢰도 평가 결과'.center(80))
print('='*80)

print("\n【 방식 1: Jaccard 유사도 기반 】")
print(f"  • 평균 Jaccard: {mean_jaccard:.4f}")
print(f"  • 완전 일치: {perfect_match}개 ({100*perfect_match/len(jaccard_similarities):.1f}%)")
print(f"  • 부분 일치: {partial_match}개 ({100*partial_match/len(jaccard_similarities):.1f}%)")
print(f"  • 불일치: {no_match}개 ({100*no_match/len(jaccard_similarities):.1f}%)")
print(f"  • Krippendorff Alpha: {alpha_jaccard:.4f}")
print(f"  • 해석: {interpretation_jaccard}")
print(f"  → 리뷰별 라벨 집합 비교 (직관적)")

print("\n【 방식 2: 51개 속성-감성 쌍 확장 방식 】")
print(f"  • Krippendorff Alpha: {alpha_exp:.4f}")
print(f"  • 해석: {interpretation_exp}")
print(f"  • 정밀도 (Precision): {prec:.4f}")
print(f"  • 재현율 (Recall): {rec:.4f}")
print(f"  • F1 점수: {f1:.4f}")
print(f"  → 51개 쌍별 0/1 판단 비교 (엄밀함)")

print("\n" + "="*80)
print("결론".center(80))
print("="*80)
print(f"""
✅ AI 라벨링 신뢰도: 상당한 수준 (Substantial Agreement)

📊 주요 지표:
   • 두 방식 모두 Alpha ≥ 0.67 (상당한 일치)
   • 정밀도 73%, 재현율 78%
   • 완전 일치 47%, 부분 일치 46%

⚠️  주의사항:
   • 완벽하지는 않음 (약 25~30% 차이 존재)
   • 특정 속성에서는 100% 일치하는 경우도 있음
   • 확장방식 Alpha는 93.6% TN으로 인한 부풀려짐 주의

💡 권장 사항:
   • 발표에서는 Jaccard 부분일치 분석과 Precision/Recall 함께 언급
   • 속성별 일치도 편차 고려하여 해석
""")